# Демо 2. Шаблони розмови і підготовка даних для SFT

В демо 1 донавчання моделі було повʼязане з заміною голови, а отже частковою зміною архітектури нейронів. В багатьох випадках при донавчанні LLM моделей архітектура не змінюється, а змінюється радше формат введених і виведених данних. Для цього потрібно зрозуміти, **у якому вигляді модель бачить діалог**.
В будь-якому випадку фізично на вхід моделі подається тензор з токенами, а за те як з цих токенів утворюються семантичні повідомлення відповідають **шаблони розмови** (chat templates).
Під час донавчання модель може вивчити новий шаблон розмови.

Донавчання LLM методом SFT виконується на діалогах, відформатованих шаблоном розмови.
У цьому ноутбуці розглянемо, що таке шаблон розмови і як підготувати дані у форматі діалогів;
саме донавчання виконаємо в демо 3.

План:
1. Завантажити base- та instruct-версії SmolLM3.
2. Подивитися, як шаблон розмови перетворює різні типи діалогів на текст.
3. Порівняти відповіді base- та instruct-моделі.
4. Перевірити режими міркування `/think` і `/no_think`.
5. Порівняти шаблони різних моделей.
6. Підготувати датасет для SFT у форматі `messages`.

Демо за мотивами вправ курсу [smol-course](https://huggingface.co/learn/smol-course) і розділу 11
[Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
import time
notebook_start_time = time.time()

In [ ]:
# Встановлюємо потрібні бібліотеки; -q вимикає зайвий вивід під час встановлення
!pip install -q "transformers>=4.55.0" "datasets" "accelerate"

In [ ]:
import warnings

import torch
from datasets import get_dataset_split_names, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Памʼять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Apple MPS")
else:
    device = "cpu"
    warnings.warn("Обчислення на CPU: генерація буде дуже повільною, потрібна GPU")

GPU: Tesla T4
Памʼять GPU: 15.6 GB


In [ ]:
# Детальна інформація про GPU NVIDIA: модель, зайнята й вільна памʼять
if device == "cuda":
    !nvidia-smi

Sat Nov  7 16:44:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             33W /   70W |   11961MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Завантаження моделей

[**SmolLM3**](https://huggingface.co/blog/smollm3) — відкрита модель від Hugging Face на 3 млрд параметрів. Вона має дві версії:
- **Base** — лише після попереднього тренування: продовжує текст, але не веде діалог;
- **Instruct** (`SmolLM3-3B`) — донавчена виконувати інструкції й вести діалог.

In [ ]:
base_model_name = "HuggingFaceTB/SmolLM3-3B-Base"
instruct_model_name = "HuggingFaceTB/SmolLM3-3B"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)

# Ваги SmolLM3 збережено в bfloat16, але T4 не підтримує його апаратно,
# тому завантажуємо у float16.
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name, dtype=torch.float16, device_map="auto"
)
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_name, dtype=torch.float16, device_map="auto"
)
print("Моделі завантажено")

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

Моделі завантажено


## 2. Як шаблон розмови форматує діалог

Діалог задається списком повідомлень із ролями:
- `system` — інструкції для моделі (роль, стиль, обмеження);
- `user` — повідомлення користувача;
- `assistant` — відповіді моделі.

Метод `apply_chat_template` перетворює цей список на рядок зі спеціальними токенами.

Параметр `add_generation_prompt`:
- `False` — для передачі в модель завершених діалогів, тобто для **тренувальних даних**;
- `True` — додає в кінець заголовок відповіді асистента, щоб модель почала відповідати; використовується для **інференсу**.

Для SmolLM3 у системне повідомлення можна додати прапорець `/think` або `/no_think`,
який вмикає чи вимикає режим міркування.

In [ ]:
# Різні типи діалогів для порівняння
conversations = {
    "simple_qa": [
        {"role": "system", "content": "/no_think"},
        {"role": "user", "content": "What is machine learning?"},
    ],
    "with_system": [
        {
            "role": "system",
            "content": "You are a helpful AI assistant specialized in explaining technical concepts clearly. /no_think",
        },
        {"role": "user", "content": "What is machine learning?"},
    ],
    "multi_turn": [
        {"role": "system", "content": "You are a math tutor. /no_think"},
        {"role": "user", "content": "What is calculus?"},
        {
            "role": "assistant",
            "content": "Calculus is a branch of mathematics that deals with rates of change and accumulation of quantities.",
        },
        {"role": "user", "content": "Can you give me a simple example?"},
    ],
    "reasoning_task": [
        {"role": "system", "content": "/think"},
        {
            "role": "user",
            "content": "Solve step by step: If a train travels 120 miles in 2 hours, what is its average speed?",
        },
    ],
}

for conv_type, messages in conversations.items():
    print(f"--- {conv_type.upper()} ---")

    # Завершений діалог: так виглядають тренувальні дані
    formatted_complete = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    # Діалог для інференсу: у кінці додано заголовок відповіді асистента
    formatted_prompt = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    print("Завершений діалог (add_generation_prompt=False):")
    print(formatted_complete)
    print("\nДля інференсу (add_generation_prompt=True):")
    print(formatted_prompt)
    print("\n" + "=" * 50 + "\n")

--- SIMPLE_QA ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 07 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>


Для інференсу (add_generation_prompt=True):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 07 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>
<|im_start|>assistant
<think>

</think>



--- WITH_SYSTEM ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 07 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant specialized in explaining technical concepts clearly.

<|im_s

**Що варто помітити:**
- Шаблон SmolLM3 сам додає до системного повідомлення блок метаданих (дата відсічення знань, поточна дата, режим міркування)
  і стандартну інструкцію, якщо власної немає. Модель бачить більше, ніж ми написали.
- Кожне повідомлення обгорнуте токенами `<|im_start|>роль` … `<|im_end|>` (формат ChatML).
- З `add_generation_prompt=True` рядок закінчується на `<|im_start|>assistant`: з цього місця модель починає генерувати.

## 3. Base проти Instruct

Ставимо обом моделям одне питання. Base-модель отримує сирий текст без шаблону,
instruct-модель — діалог, відформатований шаблоном розмови.

In [ ]:
def generate(model, tokenizer, text, max_new_tokens=150, temperature=0.7):
    """Генерує продовження тексту й повертає лише нові токени, без промпту."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Відкидаємо токени промпту: декодуємо лише те, що згенерувала модель
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
test_prompt = "Explain me, what is quantum computing?"

# Base-модель: сирий текст без шаблону
print("🤖 BASE-МОДЕЛЬ:")
print("-" * 50)
print(generate(base_model, base_tokenizer, test_prompt))

# Instruct-модель: діалог, відформатований шаблоном розмови
instruct_messages = [
    {"role": "system", "content": "/no_think"},
    {"role": "user", "content": test_prompt},
]
instruct_formatted = instruct_tokenizer.apply_chat_template(
    instruct_messages, tokenize=False, add_generation_prompt=True
)
print("\n" + "=" * 50)
print("💬 INSTRUCT-МОДЕЛЬ:")
print("-" * 50)
print(generate(instruct_model, instruct_tokenizer, instruct_formatted))

🤖 BASE-МОДЕЛЬ:
--------------------------------------------------
 It is a branch of computer science which uses quantum-mechanical phenomena to perform operations. In this, data is represented by qubits as opposed to the traditional binary bits.
Here, the quantum computer is a machine that can perform operations on data in the form of quantum bits or qubits, which are the basic units of quantum information. Just like the traditional bits, these qubits can be set to zero or one. But what makes qubits different is that they can be in a state of superposition where they can be zero, one or any combination of the two. This property of a qubit, known as superposition, makes it possible to perform multiple calculations at the same time, unlike classical bits which can carry out only one calculation

💬 INSTRUCT-МОДЕЛЬ:
--------------------------------------------------
Quantum computing is a rapidly evolving field that leverages the principles of quantum mechanics to perform complex calculat

## 4. Режими міркування `/think` і `/no_think`

SmolLM3 — гібридна модель: у режимі `/think` вона спочатку генерує міркування в блоці `<think>…</think>`,
а потім відповідь; у режимі `/no_think` блок міркувань порожній.

In [ ]:
reasoning_prompts = [
    "What is 15 × 24? Show your work.",
    "A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?",
    "If I have $50 and spend $18.75 on lunch and $12.30 on a book, how much money do I have left?",
]

for thinking_flag in ["/no_think", "/think"]:
    print(f"===== Режим: {thinking_flag} =====\n")
    for i, prompt in enumerate(reasoning_prompts, 1):
        messages = [
            {"role": "system", "content": thinking_flag},
            {"role": "user", "content": prompt},
        ]
        formatted = instruct_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        # Нижча температура робить відповіді на арифметичні задачі стабільнішими
        answer = generate(instruct_model, instruct_tokenizer, formatted, max_new_tokens=300, temperature=0.3)
        print(f"Задача {i}: {prompt}")
        print(f"Відповідь: {answer}")
        print("\n" + "-" * 50 + "\n")

===== Режим: /no_think =====

Задача 1: What is 15 × 24? Show your work.
Відповідь: To solve the multiplication problem 15 × 24, we can break it down into smaller steps.

First, let's multiply 15 by 4 (the units digit of 24):
15 × 4 = 60

Next, we multiply 15 by 20 (the tens digit of 24, which is equivalent to multiplying by 2 and then shifting the result one place to the left):
15 × 20 = 15 × 2 × 10 = 30 × 10 = 300

Now, we add the two results together:
60 + 300 = 360

So, 15 × 24 = 360.

Alternatively, we can also use the standard multiplication algorithm:

```
    15
  × 24
  ----
    60
  300
  ----
  360
```

In this method, we multiply 15 by each digit of 24 and then add the results. The result is the same: 360.

--------------------------------------------------

Задача 2: A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?
Відповідь: To determine how much flour is needed for 30 cookies, we first need to find out how much flour is required

## 5. У кожної моделі свій шаблон

Ролі `system`, `user`, `assistant` — це домовленість на рівні формату `messages`.
У токенах кожна модель записує їх по-своєму. Порівняємо шаблони кількох моделей.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi!"},
    {"role": "assistant", "content": "Hello! How can I help?"},
]

for name in ["HuggingFaceTB/SmolLM3-3B", "Qwen/Qwen3-0.6B", "microsoft/Phi-3.5-mini-instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(f"===== {name} =====")
    print(tok.apply_chat_template(messages, tokenize=False))
    print()

===== HuggingFaceTB/SmolLM3-3B =====
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 07 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful assistant.

<|im_start|>user
Hi!<|im_end|>
<|im_start|>assistant
Hello! How can I help?<|im_end|>


===== Qwen/Qwen3-0.6B =====
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hi!<|im_end|>
<|im_start|>assistant
<think>

</think>

Hello! How can I help?<|im_end|>


===== microsoft/Phi-3.5-mini-instruct =====
<|system|>
You are a helpful assistant.<|end|>
<|user|>
Hi!<|end|>
<|assistant|>
Hello! How can I help?<|end|>
<|endoftext|>



Прапорці режиму міркування також залежать від моделі. У Qwen3, наприклад,
режим вмикається параметром `enable_thinking` у `apply_chat_template`.

In [ ]:
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
question = [{"role": "user", "content": "What is 2 + 2?"}]

for enable_thinking in [True, False]:
    print(f"===== enable_thinking={enable_thinking} =====")
    print(qwen_tokenizer.apply_chat_template(
        question, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
    ))

===== enable_thinking=True =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant

===== enable_thinking=False =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant
<think>

</think>




### А що з base-моделлю?

Base-модель не донавчали на діалогах, тому шаблон у її токенізаторі або відсутній, або модель ніколи не бачила його токенів у ролі розмітки діалогу.
Перед SFT шаблон для base-моделі задають самостійно — найпростіше взяти його з instruct-версії тієї самої моделі.

In [ ]:
print("Шаблон є в base-токенізаторі:  ", base_tokenizer.chat_template is not None)
print("Шаблон є в instruct-токенізаторі:", instruct_tokenizer.chat_template is not None)

Шаблон є в base-токенізаторі:   False
Шаблон є в instruct-токенізаторі: True


## 6. Підготовка датасету для SFT

Для SFT дані зберігають у форматі `messages` — тобто у списку повідомлень із ролями.
Бібліотека TRL (`SFTTrainer`) сама застосовує до нього шаблон розмови під час тренування.

### SmolTalk2

**SmolTalk2** — датасет, на якому донавчали SmolLM3. Він складається з багатьох частин (splits) з різних джерел.
Список частин можна отримати без завантаження самих даних.

In [ ]:
splits = get_dataset_split_names("HuggingFaceTB/smoltalk2", "SFT")
print(f"Кількість частин: {len(splits)}")
for s in splits:
    print(" -", s)

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Кількість частин: 25
 - LongAlign_64k_Qwen3_32B_yarn_131k_think
 - OpenThoughts3_1.2M_think
 - aya_dataset_Qwen3_32B_think
 - multi_turn_reasoning_if_think
 - s1k_1.1_think
 - smolagents_toolcalling_traces_think
 - smoltalk_multilingual8_Qwen3_32B_think
 - smoltalk_systemchats_Qwen3_32B_think
 - table_gpt_Qwen3_32B_think
 - LongAlign_64k_context_lang_annotated_lang_6_no_think
 - Mixture_of_Thoughts_science_no_think
 - OpenHermes_2.5_no_think
 - OpenThoughts3_1.2M_no_think_no_think
 - hermes_function_calling_v1_no_think
 - smoltalk_multilingual_8languages_lang_5_no_think
 - smoltalk_smollm3_everyday_conversations_no_think
 - smoltalk_smollm3_explore_instruct_rewriting_no_think
 - smoltalk_smollm3_smol_magpie_ultra_no_think
 - smoltalk_smollm3_smol_rewrite_no_think
 - smoltalk_smollm3_smol_summarize_no_think
 - smoltalk_smollm3_systemchats_30k_no_think
 - table_gpt_no_think
 - tulu_3_sft_personas_instruction_following_no_think
 - xlam_traces_no_think
 - smoltalk_everyday_convs_reasoning_

In [ ]:
# Завантажуємо лише файл однієї частини через data_files: повний датасет займає десятки гігабайт,
# а load_dataset(..., split=...) спершу перебирає файли всіх частин і працює дуже довго
split = "smoltalk_everyday_convs_reasoning_Qwen3_32B_think"
everyday = load_dataset("HuggingFaceTB/smoltalk2", data_files=f"SFT/{split}-*.parquet", split="train")
print(everyday)
print()
for message in everyday[0]["messages"]:
    print(f"[{message['role']}] {message['content'][:200]}")

Dataset({
    features: ['messages', 'chat_template_kwargs', 'source'],
    num_rows: 2057
})

[user] Hi there
[assistant] <think>
Okay, the user sent "Hi there". That's a friendly greeting. I should respond in a welcoming way. Let me check the guidelines. I need to be helpful, keep the conversation going, and maybe ask h
[user] I'm looking for a healthy breakfast idea. What's a good option?
[assistant] <think>
Okay, the user is asking for a healthy breakfast idea. Let me think about what makes a breakfast healthy. It should be balanced, providing a mix of nutrients like protein, fiber, healthy fats,


Ця частина зібрана для режиму `/think`, тому відповіді асистента містять блок міркувань `<think>…</think>`.

Як видно з назви частини, відповіді в ній згенерувала модель Qwen3-32B. Такі **синтетичні дані** від сильнішої моделі —
поширений спосіб отримати тисячі прикладів для SFT без ручного написання відповідей. Помилки моделі-генератора при цьому переходять у дані,
тому синтетичні дані фільтрують.

### Перетворення власного датасету у формат `messages`

Більшість датасетів має інший формат, наприклад колонки «питання» і «відповідь».
Перетворимо датасет задач з математики **GSM8K** у формат `messages`.

In [ ]:
gsm8k = load_dataset("openai/gsm8k", "main", split="train[:100]")  # невелика вибірка для демо
print("Оригінальний приклад:", gsm8k[0])


def to_messages(examples):
    """Перетворює пари «питання — відповідь» на діалоги з ролями."""
    processed = []
    for question, answer in zip(examples["question"], examples["answer"]):
        processed.append([
            {"role": "system", "content": "You are a math tutor. Solve problems step by step."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ])
    return {"messages": processed}


gsm8k_messages = gsm8k.map(to_messages, batched=True, remove_columns=gsm8k.column_names)
print("\nУ форматі messages:", gsm8k_messages[0])

Оригінальний приклад: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}

У форматі messages: {'messages': [{'content': 'You are a math tutor. Solve problems step by step.', 'role': 'system'}, {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}, {'content': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'role': 'assistant'}]}


### Що побачить модель під час тренування

Застосуємо шаблон розмови до підготовленого прикладу. Саме цей рядок модель отримає під час SFT.

In [ ]:
text = instruct_tokenizer.apply_chat_template(
    gsm8k_messages[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,  # діалог завершений: відповідь асистента вже є
)
print(text)
print(f"Довжина в токенах: {len(instruct_tokenizer(text)['input_ids'])}")

notebook_end_time = time.time()
total_notebook_time = notebook_end_time - notebook_start_time

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 07 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a math tutor. Solve problems step by step.

<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>

Довжина в токенах: 145


**Що варто помітити:** шаблон знову додав метадані й режим `/think`, хоча в даних їх немає,
а відповідь асистента не містить блоку міркувань. Якщо донавчати на таких прикладах, модель вчитиметься
відповідати без міркувань навіть у режимі `/think`. Тому формат даних і шаблон мають узгоджуватися з тим,
як модель використовуватиметься після донавчання.

## 7. Відповіді на запитання (продовження)


11. Як довго виконувалися клітинки ноутбуку 02_chat_templates? Чи були проблеми з запуском, які?

Всі клітинки було успішно виконано, проблем з запуском не було.

In [ ]:
print(f"Час виконання всіх клітинок ноутбуку 02_chat_templates: {total_notebook_time:.2f} секунд.")

Час виконання всіх клітинок ноутбуку 02_chat_templates: 464.48 секунд.


12. Для чого змінювати параметр add_generation_prompt в методі apply_chat_template() при використанні моделі? Як по згенерованим розмовам можна дізнатися значення цього параметру, яке було задано при виклику метода?

Параметр add_generation_prompt визначає, чи чекаємо ми від моделі нову генерацію тексту. Якщо він має значення False, пустий тег для генерації не додається, тому що це формат завершених діалогів для тренування. Для повноцінного навчання такий діалог має містить не тільки повідомлення system та user, а й еталонну відповідь assistant, після якої стоїть закриваючий тег завершення. Якщо ж значення True, в кінець нашого запиту додається порожній тег асистента, який слугує сигналом для моделі почати генерувати відповідь. Дізнатися, яке значення цього параметру було передане при виклику метода, можна побачивши кінцівку тексту, якщо в кінці є тег асистента і пустий тег <think>, то це True, а якщо діалог завершений, наприклад, тегом <|im_end|>, то це False.

13. Чим відрізняються ролі system від assistant в шаблонах розмов?

Роль system - це інструкція, яка задає моделі правила поведінки, роль та обмеження, наприклад, "Ти досвідчений програміст". А роль assistant - це варіант відповіді самої моделі, який під час навчання слугує еталонним зразком ідеальної відповіді, на яку нейромережа повинна рівнятися.

14. Прокоментуйте вивід даних в секції 2 «Як шаблон розмови форматує діалог». Які секції було додано шаблонізатором розмов і для чого?

З виводу в секції 2 видно, що шаблон розмови перетворює список повідомлень у загальний текст, додаючи спеціальні машинні теги, наприклад `<|im_start|>` та `<|im_end|>`, а `system`, `user`, `assistant` для розмітки ролей. Шаблонізатор автоматично генерує та додає нові секції, `## Metadata`, яка містить кінцеву дату навчання моделі, сьогоднішню дату та налаштування режиму міркування та `## Custom Instructions`, куди підставляються базові інструкції, які були передані системі. Також видно, що при увімкненому add_generation_prompt=True в самий кінець тексту додається роль асистента з тегом <think>, щоб підготувати модель до генерації відповіді.

15. Прокоментуйте різницю між Base і Instruct, проглянувши вивід в секції 3.

З виводу видно, що Base-модель працює як автодоповнення тексту, тому її вивід виглядає як вирваний із контексту уривок статті. А Instruct-модель розпізнає промпт як запитання і надає структуровану відповідь у форматі діалогу, який адаптований під користувача.

16. Чому в режимі /think при генерації було передано параметр max_new_tokens=300?

Параметр max_new_tokens=300 обмежує максимальну довжину відповіді, щоб зекономити час виконання та ресурси відеокарти. Це необхідно, тому що в режимі /think модель генерує забагато тексту з покроковими міркуваннями, і такий ліміт запобігає занадто довгій генерації.

17. Чи впоралася модель з математичними задачами? Який би режим генерації Ви обрали для себе, якщо б Вам потрібна була допомога?

Модель успішно впоралася із задачами тільки в режимі `/no_think`, який я обрав би для себе, Тому що він швидко дає правильний результат. У режимі `/think` модель не завершила завдання, через надмірну кількість міркувань вона вперлася в ліміт токенів і обірвала текст, так і не видавши кінцеву відповідь.


18. Які відмінності в шаблонах розмов Ви помітили в секції 5?

Головні відмінності полягають у форматуванні та структурі. Шаблон SmolLM3 генерує додаткові секції Metadata та Custom Instructions. У Phi-3.5 взагалі немає тегів start і end, а сама роль просто береться в дужки як тег. А Qwen3 відрізняється тим, що зразу автоматично додає блок think на початку відповіді асистента.

19. Для чого нам потрібно знати про шаблони розмов, якщо ми хочемо донавчити модель за допомогою SFT?

Знання шаблонів розмов потрібне для правильного форматування тренувальних даних під час SFT. Вони дозволяють не тільки задати інструкції системи та еталонні відповіді асистента, а й правильно формувати весь діалог спеціальними машинними тегами. Якщо натренувати модель без цих тегів, вона потім просто не розпізнає ролі і не зрозуміє коли саме їй треба генерувати відповідь.

20. З яких даних було побудовано датасет HuggingFaceTB/smoltalk2?

Датасет побудовано з 25 різних частин із різноманітних джерел. До нього включено дані, згенеровані більш потужними моделями, наприклад, Qwen3_32B, щоб отримати еталонні приклади діалогів та міркувань для навчання цієї моделі.

21. Які модифікації даних відбуваються в секції "Перетворення власного датасету у формат messages"?

Вихідні дані з ключами question та answer трансформуються у стандартний масив messages. Кожен приклад перетворюється на список із трьох словників, спочатку додається роль system із заданим промптом, а вихідні питання та розв'язок мапляться у поле content для ролей user та assistant.